# Práctica 4. Detección de plagio.

In [ ]:
import sys, os
from pathlib import Path
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import pandas as pd
from itertools import islice


PROJECT_ROOT = Path.cwd().parent.parent 
sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "4°-practica" / "data"

# Importamos las funciones ya creadas para facilitar el trabajo
from scripts.text_preprocess import preprocess_text, tokenize_words, characters_no_spaces
from scripts.freq_analysis import char_frequencies, word_frequencies, top_items
from scripts.io_utils import read_text


## 0.  Creamos la función para el preprocesamiento en general (recordemos que son muchos documentos)

In [17]:
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

stemmer = PorterStemmer()

In [ ]:
def preprocess_for_plagiarism(text: str) -> list[str]:
    """
    Preprocesa el texto aplicando:
    - limpieza básica con preprocess_text()
    - tokenización con tokenize_words()
    - remoción de stopwords
    - stemming con PorterStemmer
    Devuelve una lista de tokens limpios.
    """
    # Limpieza general
    cleaned = preprocess_text(text)
    
    # Tokenización
    tokens = tokenize_words(cleaned)
    
    # Remover stopwords y aplicar stemming
    processed = [
        stemmer.stem(word) 
        for word in tokens 
        if word not in stop_words and word.isalpha()
    ]
    
    return processed

## 2. Lectura y preprocesamiento de texto

In [28]:
SOURCE_DIR = DATA_DIR / "source-documents"
SUSPICIOUS_DIR = DATA_DIR / "suspicious-documents"

def load_and_preprocess_documents(directory: Path) -> dict[str, list[str]]:
    """
    Lee todos los documentos .txt de un directorio y aplica preprocesamiento.
    Retorna un diccionario con:
        { nombre_archivo: lista_de_tokens_preprocesados }
    """
    docs = {}
    
    for file_name in os.listdir(directory):
        if file_name.endswith(".txt"):
            file_path = directory / file_name
            text = read_text(file_path)
            tokens = preprocess_for_plagiarism(text)
            docs[file_name] = tokens
            
    return docs

In [29]:
source_docs = load_and_preprocess_documents(SOURCE_DIR)
suspicious_docs = load_and_preprocess_documents(SUSPICIOUS_DIR)

print(f"Cantidad de documentos fuente: {len(source_docs)}")
print(f"Cantidad de documentos sospechosos: {len(suspicious_docs)}")

Cantidad de documentos fuente: 237
Cantidad de documentos sospechosos: 2370


In [30]:
first_source = next(iter(source_docs.items()))
print("\nEjemplo de documento fuente procesado:")
print("Nombre:", first_source[0])
print("Tokens:", first_source[1][:20])


Ejemplo de documento fuente procesado:
Nombre: source-document0001.txt
Tokens: ['commiss', 'stop', 'short', 'blame', 'chief', 'gate', 'problem', 'said', 'chief', 'serv', 'two', 'consecut', 'five', 'year', 'term', 'mr', 'gate', 'serv', 'year', 'therefor']


## 3. Cálculo de similitud.

Creamos las funciones para cada medtida de similitud.

In [ ]:
def jaccard_similarity(tokens_a: list[str], tokens_b: list[str]) -> float:
    """
    Calcula la similitud de Jaccard entre dos conjuntos de palabras.
    Fórmula:
        J(A, B) = |A ∩ B| / |A ∪ B|
    """
    set_a, set_b = set(tokens_a), set(tokens_b)
    intersection = len(set_a.intersection(set_b))
    union = len(set_a.union(set_b))
    return intersection / union if union != 0 else 0.0


def dice_similarity(tokens_a: list[str], tokens_b: list[str]) -> float:
    """
    Calcula la similitud de Dice entre dos conjuntos de palabras.
    Fórmula:
        D(A, B) = 2 * |A ∩ B| / (|A| + |B|)
    """
    set_a, set_b = set(tokens_a), set(tokens_b)
    intersection = len(set_a.intersection(set_b))
    denominator = (len(set_a) + len(set_b))
    return (2 * intersection / denominator) if denominator != 0 else 0.0

In [ ]:
def top_similar_documents(
    suspicious_docs: dict[str, list[str]],
    source_docs: dict[str, list[str]],
    top_n: int = 10,
    metric: str = "jaccard"
) -> dict[str, list[tuple[str, float]]]:
    """
    Para cada documento sospechoso, calcula la similitud con todos los documentos fuente
    usando la métrica especificada ('jaccard' o 'dice') y devuelve los top-N más similares.
    """
    results = {}
    
    for susp_name, susp_tokens in suspicious_docs.items():
        similarities = []
        
        for src_name, src_tokens in source_docs.items():
            if metric == "jaccard":
                sim = jaccard_similarity(susp_tokens, src_tokens)
            elif metric == "dice":
                sim = dice_similarity(susp_tokens, src_tokens)
            else:
                raise ValueError("La métrica debe ser 'jaccard' o 'dice'.")
            
            similarities.append((src_name, sim))
        
        # Ordenar descendente por similitud
        top_matches = sorted(similarities, key=lambda x: x[1], reverse=True)
        results[susp_name] = list(islice(top_matches, top_n))
    
    return results

## 4. Resultados finales (10 docs más parecidos para los 10 sospechosos seleccionados)

In [48]:
suspicious_subset = dict(islice(suspicious_docs.items(), 10))

In [49]:
jaccard_results = top_similar_documents(
    suspicious_subset,
    source_docs,
    top_n=10,
    metric="jaccard"
)


dice_results = top_similar_documents(
    suspicious_subset,
    source_docs,
    top_n=10,
    metric="dice"
)

def results_to_dataframe(results: dict[str, list[tuple[str, float]]], metric_name: str) -> pd.DataFrame:
    rows = []
    for susp_name, matches in results.items():
        for src_name, sim in matches:
            rows.append({
                "suspicious_document": susp_name,
                "source_document": src_name,
                f"{metric_name}_similarity": sim
            })
    return pd.DataFrame(rows)

In [50]:
df_jaccard = results_to_dataframe(jaccard_results, "jaccard")
df_dice = results_to_dataframe(dice_results, "dice")

df_final = pd.merge(
    df_jaccard,
    df_dice,
    on=["suspicious_document", "source_document"],
    how="outer"
)

In [51]:
df_final = df_final.sort_values(
    by=["suspicious_document", "jaccard_similarity"],
    ascending=[True, False]
).reset_index(drop=True)

df_final.head(20)

,suspicious_document,source_document,jaccard_similarity,dice_similarity
0,suspicious-document0001.txt,source-document0001.txt,0.152685,0.264920
1,suspicious-document0001.txt,source-document0040.txt,0.142667,0.249708
2,suspicious-document0001.txt,source-document0060.txt,0.127848,0.226712
3,suspicious-document0001.txt,source-document0014.txt,0.126812,0.225080
4,suspicious-document0001.txt,source-document0104.txt,0.120452,0.215006
5,suspicious-document0001.txt,source-document0210.txt,0.117747,0.210687
6,suspicious-document0001.txt,source-document0152.txt,0.117397,0.210127
7,suspicious-document0001.txt,source-document0023.txt,0.117338,0.210031
8,suspicious-document0001.txt,source-document0072.txt,0.116906,0.209340
9,suspicious-document0001.txt,source-document0149.txt,0.116732,0.209059
